# TweetAge v8: modelo de lenguaje jerarquico (clase respaldada en un modelo global)

**Objetivo.** Prueba dentro de los modelos de lenguaje propios: cada modelo de clase deja de tener su propio Kneser-Ney y se respalda en un modelo global entrenado por el equipo con todos los tweets del ajuste. La idea es que las clases con menos evidencia en un historial hereden la estimacion global en lugar de suavizar solo con sus propios datos.

**Modelo global G.** El ModifiedKN de v6 entrenado con todos los tweets del pliegue de ajuste, sin etiquetas: caracteres de orden 4 y palabras de orden 3.

**Modelo de clase c.** Para el orden n del nivel (4 en caracteres, 3 en palabras) y cada historial h de n-1 simbolos, con n_c(hw) el conteo de la secuencia hw en los tweets de la clase c y n_c(h) = suma sobre w de n_c(hw):

P_c(w | h) = (n_c(hw) + beta * P_G(w | h)) / (n_c(h) + beta)

donde P_G es la probabilidad del modelo global del mismo orden. Es un respaldo tipo Dirichlet (MacKay y Peto, 1995): beta actua como numero de pseudo-observaciones tomadas del global. Si la clase no vio el historial, P_c = P_G. Como P_G suma 1 sobre el vocabulario, P_c tambien. Sigue siendo un modelo de lenguaje propio y generativo: no hay clasificador discriminativo, vectores de caracteristicas, embeddings ni pesos preentrenados.

**Valores fijados antes de ejecutar.** beta = 10 y beta = 50; no se prueban otros.

**Puntuacion.** score = 0,75 x log P_c(caracteres) + 0,25 x log P_c(palabras) + log prior, la misma combinacion de A. Sin calibracion de sesgos. Normalizacion A de v6 sin cambios.

**Referencia A** (recalculada en esta corrida): ModifiedKN por clase, caracteres orden 4, palabras orden 3, lambda 0,75, mas log prior. Valores esperados en v7_lower: F1 OOF 0,440081; por pliegue 0,440843 / 0,432485 / 0,440697 / 0,447354 / 0,439014. Si el recalculo no coincide dentro de 1e-6, la corrida se detiene.

**Regla de decision (fijada antes de ejecutar).** Un beta se acepta solo si la media de las cinco diferencias pareadas de F1 macro contra A es > 0 y mayor que su desviacion estandar muestral (ddof=1). Entre aceptados se elige la mayor diferencia media; en empate, el beta menor. Si ninguno se acepta: "NO PASA", sin holdout ni CSV. Si se acepta: holdout evaluado una sola vez y `age_submission_v8_lm_jerarquico.csv` con escritura exclusiva. Heuristica, no prueba formal.


In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED=42
np.random.seed(SEED)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
CHAR_ORDERS=[4]
WORD_ORDERS=[3]
CHAR_ORDER,WORD_ORDER,CHAR_WEIGHT=4,3,0.75
BETAS=[10.0,50.0]
REF_FOLD_F1=np.array([0.440843,0.432485,0.440697,0.447354,0.439014])
REF_OOF_F1=0.440081
REF_TOLERANCE=1e-6
NORMALIZATION_TOLERANCE=1e-9
roots=[Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR=next((candidate for candidate in roots if (candidate/'TweetAge_v8_lm_jerarquico.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH=OUT_DIR/'TweetAge_v8_lm_jerarquico.ipynb'
SUB_PATH=OUT_DIR/'age_submission_v8_lm_jerarquico.csv'
SPLIT_PATH=OUT_DIR/'split_idx_v7_lower_seed42.npz'
DATA_DIR=OUT_DIR/'tweet-age-2026-puj'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
INPUT_HASHES={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in OUT_DIR.iterdir() if path.is_file() and path!=NOTEBOOK_PATH}
assert PROTECTED_HASHES[SPLIT_PATH.name]=='692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83','Hash del NPZ v7_lower distinto; detener.'
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Semilla',SEED,'| notebook',NOTEBOOK_PATH.name,'| betas fijados',BETAS)
print('SHA-256 NPZ v7_lower:',PROTECTED_HASHES[SPLIT_PATH.name])
print('SHA-256 age_train.csv:',INPUT_HASHES['age_train.csv'])

def check_compliance(notebook):
    # Guardia AST de v6 mas prohibiciones de esta ronda: vectorizadores, NB de sklearn, matrices dispersas, embeddings, preentrenados, torch y clasificadores discriminativos.
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove', 'gensim', 'pretrained', 'from_pretrained', 'multinomialnb', 'complementnb', 'bernoullinb', 'gaussiannb', 'naive_bayes', 'sparse', 'csr_matrix', 'csc_matrix', 'coo_matrix', 'lil_matrix', 'torch', 'svm', 'svc', 'linearsvc', 'ensemble', 'tree', 'neighbors', 'neural_network', 'mlpclassifier', 'logisticregression', 'sgdclassifier', 'discriminant_analysis', 'xgboost', 'lightgbm', 'catboost'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, embeddings, preentrenados, torch ni clasificadores discriminativos.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Semilla 42 | notebook TweetAge_v8_lm_jerarquico.ipynb | betas fijados [10.0, 50.0]
SHA-256 NPZ v7_lower: 692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83
SHA-256 age_train.csv: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, embeddings, preentrenados, torch ni clasificadores discriminativos.


## Datos, normalizacion A y verificacion del NPZ v7_lower
Celda identica a la de v7 hiperparametros: hashes del train y de los grupos, cero fuga en minusculas en holdout y en los cinco pliegues.

In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns=verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw=pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean.loc[~excluded]
train=train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
train['normalized']=train.text.map(normalize_text)
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:
    split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==INPUT_HASHES['age_train.csv'] and int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_keys=np.array([text.lower() for text in train.normalized],dtype=object)
groups_hash=hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert groups_hash==str(split['groups_sha256'])
group_ids,unique_keys=pd.factorize(group_keys,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert set(group_ids[idx_development])==set(dev_groups) and set(group_ids[idx_holdout])==set(holdout_groups)
assert not np.intersect1d(dev_groups,holdout_groups).size
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
group_counts=np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
texts_development=train.normalized.iloc[idx_development].to_numpy()
y_development=y_all[idx_development]
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[dict(validacion='holdout',textos_compartidos=0)]
for fold_number,(fit_positions,app_positions) in enumerate(group_folds,1):
    fit_idx=np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx=np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    shared={text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    leak_rows.append(dict(validacion=str(fold_number),textos_compartidos=len(shared)))
    if shared:
        print('DETENER: fuga en pliegue',fold_number,'ejemplos',sorted(shared)[:10])
        raise RuntimeError('Fuga en minusculas; no entrenar.')
    folds.append((fit_idx,app_idx))
print('NPZ REUTILIZADO SIN REGENERAR:',SPLIT_PATH.name)
print('SHA-256 train y grupos correctos. SHA-256 grupos:',groups_hash)
print('Filas train original/limpio/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Textos solo URL:',int(train.normalized.str.strip().eq('<url>').sum()),'solo usuario:',int(train.normalized.str.strip().eq('<user>').sum()))
display(pd.DataFrame(leak_rows))


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ REUTILIZADO SIN REGENERAR: split_idx_v7_lower_seed42.npz
SHA-256 train y grupos correctos. SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Filas train original/limpio/excluidas: 169336 169181 155
Desarrollo grupos/filas: 126309 132987 holdout grupos/filas: 31184 36194
Textos solo URL: 3785 solo usuario: 287


,validacion,textos_compartidos
0,holdout,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


## Kneser-Ney modificado de v6 (referencia A y modelo global G)

In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def test_shared_counts():
    toy_texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(toy_texts,level)
        sequences=[encode_text(text,level,mapping) for text in toy_texts]
        support=set(mapping.values())
        shared_model=ModifiedKN(max(orders),support).fit(sequences)
        for order in orders:
            separate_model=ModifiedKN(order,support).fit(sequences)
            for sequence in sequences+[UNK]:
                assert abs(shared_model.log_probability(sequence,order)-separate_model.log_probability(sequence,order))<1e-12,(level,order)
            separate_model.close()
        shared_model.close()
    print('Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.')

test_modified_kn()
test_shared_counts()
timing_rows=[]

def fit_score_orders(fit_texts,fit_labels,app_texts,level,orders,phase,fold):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    preprocessing=time.perf_counter()-started
    matrices={order:np.zeros((len(app_texts),6)) for order in orders}
    fit_seconds=0.0
    score_seconds={order:0.0 for order in orders}
    for class_index in range(6):
        fit_started=time.perf_counter()
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        fit_seconds+=time.perf_counter()-fit_started
        for order in orders:
            score_started=time.perf_counter()
            matrices[order][:,class_index]=[model.log_probability(sequence,order) for sequence in encoded_app]
            score_seconds[order]+=time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
    for order in orders:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,orden=order,ajuste_compartido_s=preprocessing+fit_seconds,puntuacion_s=score_seconds[order],tiempo_amortizado_s=score_seconds[order]+(preprocessing+fit_seconds)/len(orders)))
    print(phase,fold,level,'ordenes',orders,'segundos',round(time.perf_counter()-started,2),flush=True)
    return matrices


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.


## Modelo jerarquico y pruebas

In [4]:
def class_gram_counts(sequences,order):
    # n_c(hw) al orden maximo, con el mismo relleno BOS que el ModifiedKN, y n_c(h) = suma sobre w de n_c(hw).
    padding=BOS*(order-1)
    counts=Counter()
    for sequence in sequences:
        padded=padding+sequence+EOS
        counts.update(padded[position-order+1:position+1] for position in range(order-1,len(padded)))
    totals=Counter()
    for gram,count in counts.items():
        totals[gram[:-1]]+=count
    return counts,totals

def hierarchical_probability(gram,counts,totals,global_probability,beta):
    return (counts.get(gram,0)+beta*global_probability(gram))/(totals.get(gram[:-1],0)+beta)

def normalization_rows(counts,totals,global_model,histories,betas,level,label):
    symbols=sorted(global_model.support)
    rows=[]
    for history in histories:
        global_values=np.array([global_model.probability(history+symbol) for symbol in symbols])
        class_values=np.array([counts.get(history+symbol,0) for symbol in symbols],dtype=float)
        total=totals.get(history,0)
        for beta in betas:
            mass=float(((class_values+beta*global_values)/(total+beta)).sum())
            rows.append(dict(nivel=level,clase=label,historial=repr(history),n_c_h=total,beta=beta,suma=mass,error=abs(mass-1),suma_global=float(global_values.sum())))
    return rows

def hierarchical_scores(fit_texts,fit_labels,app_texts,level,order,betas,check_normalization=False):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    global_model=ModifiedKN(order,set(mapping.values())).fit(encoded_fit)
    padding=BOS*(order-1)
    gram_index,sequence_positions,gram_positions={},[],[]
    for row,sequence in enumerate(encoded_app):
        padded=padding+sequence+EOS
        for position in range(order-1,len(padded)):
            sequence_positions.append(row)
            gram_positions.append(gram_index.setdefault(padded[position-order+1:position+1],len(gram_index)))
    grams=list(gram_index)
    sequence_positions=np.array(sequence_positions)
    gram_positions=np.array(gram_positions)
    global_values=np.array([global_model.probability(gram) for gram in grams])
    numerators=np.zeros((len(grams),6))
    denominators=np.zeros((len(grams),6))
    checks=[]
    for class_index in range(6):
        counts,totals=class_gram_counts([sequence for sequence,label in zip(encoded_fit,fit_labels) if label==class_index],order)
        numerators[:,class_index]=[counts.get(gram,0) for gram in grams]
        denominators[:,class_index]=[totals.get(gram[:-1],0) for gram in grams]
        if check_normalization and class_index in (0,5):
            frequent=[history for history,_ in totals.most_common(2)]
            unseen=next((gram[:-1] for gram in grams if gram[:-1] not in totals),None)
            middle=sorted(totals)[len(totals)//2]
            histories=list(dict.fromkeys([padding,*frequent,middle]+([unseen] if unseen is not None else [])))
            checks.extend(normalization_rows(counts,totals,global_model,histories,betas,level,CLASSES[class_index]))
        del counts,totals
    scores={}
    for beta in betas:
        log_probabilities=np.log((numerators+beta*global_values[:,None])/(denominators+beta))
        scores[beta]=np.column_stack([np.bincount(sequence_positions,weights=log_probabilities[gram_positions,class_index],minlength=len(app_texts)) for class_index in range(6)])
    global_model.close()
    del global_model
    gc.collect()
    print(level,'orden',order,'gramas distintos en evaluacion',len(grams),'segundos',round(time.perf_counter()-started,2),flush=True)
    return scores,checks

def test_hierarchical():
    texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!','chao chao']
    labels=np.array([0,0,1,1,2,2])
    for level,order in [('char',4),('word',3)]:
        mapping=make_encoder(texts,level)
        encoded=[encode_text(text,level,mapping) for text in texts]
        global_model=ModifiedKN(order,set(mapping.values())).fit(encoded)
        counts,totals=class_gram_counts([sequence for sequence,label in zip(encoded,labels) if label==0],order)
        histories=list(totals)[:3]+[BOS*(order-1),UNK*(order-1)]
        rows=normalization_rows(counts,totals,global_model,histories,BETAS,level,'toy')
        assert max(row['error'] for row in rows)<NORMALIZATION_TOLERANCE
        # Puntuacion vectorizada == suma directa gram a gram.
        app=['hola amiga','adios!']
        scores,_=hierarchical_scores(texts,labels,app,level,order,BETAS)
        for beta in BETAS:
            for row,text in enumerate(app):
                padded=BOS*(order-1)+encode_text(text,level,mapping)+EOS
                direct=sum(math.log(hierarchical_probability(padded[position-order+1:position+1],counts,totals,global_model.probability,beta)) for position in range(order-1,len(padded)))
                assert abs(scores[beta][row,0]-direct)<1e-9,(level,beta,text)
        # Historial no visto por la clase: P_c = P_G.
        unseen_gram=UNK*order
        assert abs(hierarchical_probability(unseen_gram,counts,totals,global_model.probability,10.0)-global_model.probability(unseen_gram))<1e-15
        global_model.close()
    print('Prueba jerarquica: normalizacion 1e-9, P_c = P_G en historial no visto y puntuacion vectorizada igual a la suma directa.')

test_hierarchical()


char orden 4 gramas distintos en evaluacion 18 segundos 0.02


word orden 3 gramas distintos en evaluacion 6 segundos 0.02


Prueba jerarquica: normalizacion 1e-9, P_c = P_G en historial no visto y puntuacion vectorizada igual a la suma directa.


## Historial de intentos

In [5]:
history=pd.DataFrame([
    dict(intento='v5 KN (particion con fuga)',f1_oof='0,43847 (calibrado)',estado='Kaggle 0,45643; holdout inflado'),
    dict(intento='v6 KN modificado (agrupada)',f1_oof='0,44106 sin calibrar / 0,44353 calibrado',estado='Valida publicada; Kaggle 0,45737'),
    dict(intento='Referencia A en v7_lower',f1_oof='0,440081',estado='Referencia de v7 y v8'),
    dict(intento='v7 ordinal',f1_oof='sin salida guardada',estado='NO PASA'),
    dict(intento='v7 descuento con concentracion',f1_oof='0,440875',estado='NO PASA'),
    dict(intento='v7 normalizacion (A/B/C)',f1_oof='0,440081 / 0,439709 / 0,436893',estado='NO PASA'),
    dict(intento='v7 hiperparametros (mejor C7)',f1_oof='0,437677',estado='NO PASA'),
    dict(intento='v7 LM neuronal GRU',f1_oof='pliegue 1: 0,411131',estado='NO PASA EN ETAPA 1'),
    dict(intento='v8 Naive Bayes',f1_oof='0,376727',estado='NO PASA'),
    dict(intento='v8 PPM-C',f1_oof='0,421730',estado='NO PASA'),
    dict(intento='v8 LM jerarquico',f1_oof='(se llena al final)',estado='(se llena al final)'),
])
print('Historial de intentos (F1 macro OOF sin calibrar salvo indicacion). Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.')
display(history)


Historial de intentos (F1 macro OOF sin calibrar salvo indicacion). Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.


,intento,f1_oof,estado
0,v5 KN (particion con fuga),"0,43847 (calibrado)","Kaggle 0,45643; holdout inflado"
1,v6 KN modificado (agrupada),"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737"
2,Referencia A en v7_lower,"0,440081",Referencia de v7 y v8
3,v7 ordinal,sin salida guardada,NO PASA
4,v7 descuento con concentracion,"0,440875",NO PASA
5,v7 normalizacion (A/B/C),"0,440081 / 0,439709 / 0,436893",NO PASA
6,v7 hiperparametros (mejor C7),"0,437677",NO PASA
7,v7 LM neuronal GRU,"pliegue 1: 0,411131",NO PASA EN ETAPA 1
8,v8 Naive Bayes,"0,376727",NO PASA
9,v8 PPM-C,"0,421730",NO PASA


## Validacion cruzada: referencia A recalculada, beta = 10 y beta = 50

In [6]:
def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,labels=np.arange(6),average='macro',zero_division=0)

cv_started=time.perf_counter()
keys=['REF']+[f'beta_{beta:g}' for beta in BETAS]
oof={key:np.zeros((len(y_development),6)) for key in keys}
fold_rows,normalization_checks=[],[]
for fold_number,(fit_idx,app_idx) in enumerate(folds,1):
    started=time.perf_counter()
    fit_texts,app_texts=texts_development[fit_idx],texts_development[app_idx]
    fit_labels=y_development[fit_idx]
    assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    characters=fit_score_orders(fit_texts,fit_labels,app_texts,'char',[CHAR_ORDER],'CV',fold_number)[CHAR_ORDER]
    words=fit_score_orders(fit_texts,fit_labels,app_texts,'word',[WORD_ORDER],'CV',fold_number)[WORD_ORDER]
    oof['REF'][app_idx]=CHAR_WEIGHT*characters+(1-CHAR_WEIGHT)*words+prior
    hier_characters,checks_c=hierarchical_scores(fit_texts,fit_labels,app_texts,'char',CHAR_ORDER,BETAS,check_normalization=fold_number==1)
    hier_words,checks_w=hierarchical_scores(fit_texts,fit_labels,app_texts,'word',WORD_ORDER,BETAS,check_normalization=fold_number==1)
    normalization_checks.extend(checks_c+checks_w)
    for beta in BETAS:
        oof[f'beta_{beta:g}'][app_idx]=CHAR_WEIGHT*hier_characters[beta]+(1-CHAR_WEIGHT)*hier_words[beta]+prior
    y_app=y_development[app_idx]
    row=dict(pliegue=fold_number,**{f'f1_{key}':macro_f1(y_app,oof[key][app_idx].argmax(axis=1)) for key in keys},segundos=time.perf_counter()-started)
    fold_rows.append(row)
    print(f'pliegue {fold_number}: '+' | '.join(f'{key} {row["f1_"+key]:.6f}' for key in keys)+f' | {row["segundos"]:.1f} s',flush=True)
    del characters,words,hier_characters,hier_words
    gc.collect()

normalization_table=pd.DataFrame(normalization_checks)
print('VERIFICACION DE NORMALIZACION (pliegue 1, clases 13-17 y 65-xx, ambos niveles y ambos beta):',len(normalization_table),'historiales x beta; error maximo',float(normalization_table.error.max()))
display(normalization_table)
assert normalization_table.error.max()<NORMALIZATION_TOLERANCE and set(normalization_table.nivel)=={'char','word'} and set(normalization_table.beta)==set(BETAS)
print('NORMALIZACION: PASA, P_c(.|h) suma 1 con tolerancia 1e-9.')

folds_table=pd.DataFrame(fold_rows)
reference_oof=macro_f1(y_development,oof['REF'].argmax(axis=1))
reference_gap=np.abs(folds_table.f1_REF.to_numpy()-REF_FOLD_F1)
print('Referencia A recalculada por pliegue:',folds_table.f1_REF.round(6).tolist(),'| OOF',round(reference_oof,6))
print('Max |diferencia| con la referencia medida por pliegue:',float(reference_gap.max()),'| OOF:',abs(reference_oof-REF_OOF_F1))
if reference_gap.max()>REF_TOLERANCE or abs(reference_oof-REF_OOF_F1)>REF_TOLERANCE:
    raise RuntimeError('DETENER: la referencia A recalculada no coincide con la medida (tolerancia 1e-6).')
print('REFERENCIA A VERIFICADA (tolerancia 1e-6).')

result_rows=[]
for key in keys:
    per_fold=folds_table[f'f1_{key}'].to_numpy()
    differences=per_fold-folds_table.f1_REF.to_numpy()
    result_rows.append(dict(modelo='A (KN por clase)' if key=='REF' else f'jerarquico beta={key.split("_")[1]}',beta=np.nan if key=='REF' else float(key.split('_')[1]),f1_macro_oof=macro_f1(y_development,oof[key].argmax(axis=1)),**{f'pliegue_{number}':value for number,value in enumerate(per_fold,1)},f1_pliegues_std=float(per_fold.std(ddof=1)),diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))))
results=pd.DataFrame(result_rows)
print('TABLA: F1 macro OOF sin calibrar, F1 por pliegue, desviacion por pliegue y diferencia pareada contra A (ddof=1)')
print(results.to_string(index=False))
print('CV segundos:',round(time.perf_counter()-cv_started,1))


CV 1 char ordenes [4] segundos 7.09


CV 1 word ordenes [3] segundos 5.9


char orden 4 gramas distintos en evaluacion 138601 segundos 4.75


word orden 3 gramas distintos en evaluacion 294595 segundos 4.37


pliegue 1: REF 0.440843 | beta_10 0.413861 | beta_50 0.408880 | 22.2 s


...

word orden 3 gramas distintos en evaluacion 292566 segundos 3.98


pliegue 5: REF 0.439014 | beta_10 0.413187 | beta_50 0.411240 | 21.0 s


VERIFICACION DE NORMALIZACION (pliegue 1, clases 13-17 y 65-xx, ambos niveles y ambos beta): 34 historiales x beta; error maximo 2.1760371282653068e-14


,nivel,clase,historial,n_c_h,beta,suma,error,suma_global
0,char,13-17,'\x02\x02\x02',18334,10.0,1.0,2.220446e-16,1.0
1,char,13-17,'\x02\x02\x02',18334,50.0,1.0,0.000000e+00,1.0
2,char,13-17,'āĝŖ',17768,10.0,1.0,1.110223e-16,1.0
3,char,13-17,'āĝŖ',17768,50.0,1.0,0.000000e+00,1.0
4,char,13-17,'ĴīĢ',8,10.0,1.0,0.000000e+00,1.0
5,char,13-17,'ĴīĢ',8,50.0,1.0,2.220446e-16,1.0
6,char,13-17,'\x02ĶŅ',0,10.0,1.0,0.000000e+00,1.0
7,char,13-17,'\x02ĶŅ',0,50.0,1.0,0.000000e+00,1.0
8,char,65-xx,'\x02\x02\x02',16886,10.0,1.0,2.220446e-16,1.0
9,char,65-xx,'\x02\x02\x02',16886,50.0,1.0,2.220446e-16,1.0


NORMALIZACION: PASA, P_c(.|h) suma 1 con tolerancia 1e-9.
Referencia A recalculada por pliegue: [0.440843, 0.432485, 0.440697, 0.447354, 0.439014] | OOF 0.440081
Max |diferencia| con la referencia medida por pliegue: 4.232924309954811e-07 | OOF: 2.786557278255941e-07
REFERENCIA A VERIFICADA (tolerancia 1e-6).
TABLA: F1 macro OOF sin calibrar, F1 por pliegue, desviacion por pliegue y diferencia pareada contra A (ddof=1)
            modelo  beta  f1_macro_oof  pliegue_1  pliegue_2  pliegue_3  pliegue_4  pliegue_5  f1_pliegues_std  diferencia_pareada_media  diferencia_pareada_std
  A (KN por clase)   NaN      0.440081   0.440843   0.432485   0.440697   0.447354   0.439014         0.005308                  0.000000                0.000000
jerarquico beta=10  10.0      0.414583   0.413861   0.406616   0.416387   0.422720   0.413187         0.005818                 -0.025525                0.001072
jerarquico beta=50  50.0      0.410969   0.408880   0.403753   0.413435   0.417273   0.411240 

## Decision

In [7]:
candidates=results.loc[results.beta.notna()].copy()
accepted=candidates.loc[(candidates.diferencia_pareada_media>0)&(candidates.diferencia_pareada_media>candidates.diferencia_pareada_std)].sort_values(['diferencia_pareada_media','beta'],ascending=[False,True])
PASA=not accepted.empty
selected_beta=float(accepted.iloc[0].beta) if PASA else None
best_beta=float(candidates.sort_values(['diferencia_pareada_media','beta'],ascending=[False,True]).iloc[0].beta)
for _,row in candidates.iterrows():
    print(f'beta={row.beta:g}: diferencia pareada media {row.diferencia_pareada_media:+.6f} | desviacion estandar {row.diferencia_pareada_std:.6f} | {"acepta" if row.beta in set(accepted.beta) else "no acepta"}')
print('Regla fijada: media >0 y > desviacion estandar muestral; desempate por beta menor.')
print('PASA' if PASA else 'NO PASA')
print('Beta elegido:',selected_beta if PASA else 'ninguno','| mejor beta descriptivo (mayor diferencia media):',best_beta)
if not PASA:
    print('NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v8_lm_jerarquico.csv.')


beta=10: diferencia pareada media -0.025525 | desviacion estandar 0.001072 | no acepta
beta=50: diferencia pareada media -0.029163 | desviacion estandar 0.001898 | no acepta
Regla fijada: media >0 y > desviacion estandar muestral; desempate por beta menor.
NO PASA
Beta elegido: ninguno | mejor beta descriptivo (mayor diferencia media): 10.0
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v8_lm_jerarquico.csv.


## Holdout (solo si PASA)

In [8]:
def jerarquico_final_scores(fit_texts,fit_labels,app_texts,beta):
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    characters,_=hierarchical_scores(fit_texts,fit_labels,app_texts,'char',CHAR_ORDER,[beta])
    words,_=hierarchical_scores(fit_texts,fit_labels,app_texts,'word',WORD_ORDER,[beta])
    return CHAR_WEIGHT*characters[beta]+(1-CHAR_WEIGHT)*words[beta]+prior

if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    texts_holdout=train.normalized.iloc[idx_holdout].to_numpy()
    assert not {text.lower() for text in texts_development}&{text.lower() for text in texts_holdout}
    holdout_pred=jerarquico_final_scores(texts_development,y_development,texts_holdout,selected_beta).argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=macro_f1(y_holdout,holdout_pred)
    print('F1 macro holdout v7_lower:',holdout_f1,'| beta',selected_beta)
    display(pd.DataFrame(classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)).T.loc[CLASSES,['precision','recall','f1-score','support']])
    print('Holdout segundos:',round(time.perf_counter()-started,1),'; evaluacion unica. No comparar directamente con el holdout de v6 ni con Kaggle.')
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.


## CSV (solo si PASA)

In [9]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts=test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    predictions=jerarquico_final_scores(train.normalized.to_numpy(),y_all,test_texts,selected_beta).argmax(axis=1)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[predictions]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    print('CSV VERIFICADO:',SUB_PATH.name,len(submission),'filas; coma y sin BOM.')
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.')


CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.


## F1 por clase y matriz de confusion del mejor beta

In [10]:
best_key=f'beta_{best_beta:g}'
best_predictions=oof[best_key].argmax(axis=1)
reference_predictions=oof['REF'].argmax(axis=1)
per_class=pd.DataFrame({
    'f1_A':f1_score(y_development,reference_predictions,labels=np.arange(6),average=None,zero_division=0),
    f'f1_beta_{best_beta:g}':f1_score(y_development,best_predictions,labels=np.arange(6),average=None,zero_division=0),
    'soporte_desarrollo':np.bincount(y_development,minlength=6),
},index=CLASSES)
per_class['diferencia']=per_class[f'f1_beta_{best_beta:g}']-per_class.f1_A
print('F1 por clase (OOF) de A y del mejor beta =',best_beta)
display(per_class)
matrix=confusion_matrix(y_development,best_predictions,labels=np.arange(6))
print('Matriz de confusion OOF del mejor beta =',best_beta)
display(pd.DataFrame(matrix,index=[f'real {name}' for name in CLASSES],columns=[f'pred {name}' for name in CLASSES]))
adjacent_share=sum(matrix[i,j] for i in range(6) for j in range(6) if abs(i-j)==1)/max(1,matrix.sum()-matrix.trace())
edge=per_class.loc[['13-17','65-xx'],'diferencia']
print(f'Rangos 13-17 y 65-xx: diferencia de F1 contra A = {edge["13-17"]:+.4f} y {edge["65-xx"]:+.4f}.')
print(f'HIPOTESIS (no verificada): {adjacent_share:.1%} de los errores caen en rangos adyacentes. Si los rangos extremos no mejoran con el respaldo global, su dificultad no vendria de escasez de evidencia por historial sino de que su estilo se solapa con el de los rangos vecinos. Requiere analisis cualitativo antes de afirmarlo.')


F1 por clase (OOF) de A y del mejor beta = 10.0


,f1_A,f1_beta_10,soporte_desarrollo,diferencia
13-17,0.502141,0.475475,22792,-0.026667
18-24,0.381955,0.368795,22552,-0.013160
25-34,0.343792,0.317523,22405,-0.026269
35-49,0.370839,0.343178,22168,-0.027662
50-64,0.495927,0.468541,22023,-0.027386
65-xx,0.545829,0.513989,21047,-0.031841


Matriz de confusion OOF del mejor beta = 10.0


,pred 13-17,pred 18-24,pred 25-34,pred 35-49,pred 50-64,pred 65-xx
real 13-17,10469,5923,2688,1425,1309,978
real 18-24,4856,8826,3886,1959,1671,1354
real 25-34,2746,4958,6875,3056,2639,2131
real 35-49,1552,2773,3684,6978,3973,3208
real 50-64,855,1460,2208,2889,10876,3735
real 65-xx,766,1372,1558,2192,3934,11225


Rangos 13-17 y 65-xx: diferencia de F1 contra A = -0.0267 y -0.0318.
HIPOTESIS (no verificada): 52.6% de los errores caen en rangos adyacentes. Si los rangos extremos no mejoran con el respaldo global, su dificultad no vendria de escasez de evidencia por historial sino de que su estilo se solapa con el de los rangos vecinos. Requiere analisis cualitativo antes de afirmarlo.


## Resumen

In [11]:
best_row=results.loc[results.beta==best_beta].iloc[0]
history.loc[history.intento=='v8 LM jerarquico',['f1_oof','estado']]=[f'{best_row.f1_macro_oof:.6f} (beta {best_beta:g})','PASA' if PASA else 'NO PASA']
display(history)
beta_lines='; '.join(f'beta={row.beta:g}: OOF {row.f1_macro_oof:.6f}, diferencia media {row.diferencia_pareada_media:+.6f} (desviacion {row.diferencia_pareada_std:.6f})' for _,row in results.loc[results.beta.notna()].iterrows())
summary_text=f'''## Resumen final: modelo de lenguaje jerarquico

Resultado: **{'PASA' if PASA else 'NO PASA'}**. Referencia A recalculada: F1 OOF {reference_oof:.6f}. {beta_lines}. {'Holdout evaluado una sola vez con beta ' + format(selected_beta, 'g') + ': F1 = ' + format(holdout_f1, '.6f') + '; CSV generado con escritura exclusiva.' if PASA else 'Holdout y CSV omitidos explicitamente por la regla.'}

**Idea.** Cada modelo de clase estima P_c(w|h) = (n_c(hw) + beta P_G(w|h)) / (n_c(h) + beta): usa sus propios conteos cuando tiene evidencia y se respalda en un modelo global (ModifiedKN de v6 entrenado con todos los tweets del ajuste, sin etiquetas) cuando no la tiene. Sigue siendo un modelo de lenguaje propio y generativo; la clasificacion es por verosimilitud mas log prior, sin clasificador discriminativo, vectores de caracteristicas, embeddings ni pesos preentrenados.

**Optimismo de seleccion:** elegir entre dos valores de beta introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias confirmadas: v5 calibrado 0,43847, v6 sin calibrar 0,44106, v6 calibrado 0,44353 (particiones distintas) y referencia A en v7_lower 0,440081.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Normalizacion A de v6. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.
'''
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


,intento,f1_oof,estado
0,v5 KN (particion con fuga),"0,43847 (calibrado)","Kaggle 0,45643; holdout inflado"
1,v6 KN modificado (agrupada),"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737"
2,Referencia A en v7_lower,"0,440081",Referencia de v7 y v8
3,v7 ordinal,sin salida guardada,NO PASA
4,v7 descuento con concentracion,"0,440875",NO PASA
5,v7 normalizacion (A/B/C),"0,440081 / 0,439709 / 0,436893",NO PASA
6,v7 hiperparametros (mejor C7),"0,437677",NO PASA
7,v7 LM neuronal GRU,"pliegue 1: 0,411131",NO PASA EN ETAPA 1
8,v8 Naive Bayes,"0,376727",NO PASA
9,v8 PPM-C,"0,421730",NO PASA


## Resumen final: modelo de lenguaje jerarquico

Resultado: **NO PASA**. Referencia A recalculada: F1 OOF 0.440081. beta=10: OOF 0.414583, diferencia media -0.025525 (desviacion 0.001072); beta=50: OOF 0.410969, diferencia media -0.029163 (desviacion 0.001898). Holdout y CSV omitidos explicitamente por la regla.

**Idea.** Cada modelo de clase estima P_c(w|h) = (n_c(hw) + beta P_G(w|h)) / (n_c(h) + beta): usa sus propios conteos cuando tiene evidencia y se respalda en un modelo global (ModifiedKN de v6 entrenado con todos los tweets del ajuste, sin etiquetas) cuando no la tiene. Sigue siendo un modelo de lenguaje propio y generativo; la clasificacion es por verosimilitud mas log prior, sin clasificador discriminativo, vectores de caracteristicas, embeddings ni pesos preentrenados.

**Optimismo de seleccion:** elegir entre dos valores de beta introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias confirmadas: v5 calibrado 0,43847, v6 sin calibrar 0,44106, v6 calibrado 0,44353 (particiones distintas) y referencia A en v7_lower 0,440081.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Normalizacion A de v6. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.


HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
